The code for retrieving and structuring the data is adapted from MET's Python example for Frost [1].

In [ ]:
from pathlib import Path

import requests
import pandas as pd

client_id = Path("frost_client_id.txt").read_text().strip()

In [ ]:
# Define endpoint and parameters
endpoint = 'https://frost.met.no/observations/v0.jsonld'
parameters = {
    'sources': 'SN17850',
    'elements': 'mean(air_temperature P1D)',
    'referencetime': '2025-01-01/2025-12-31',
    }

# Issue an HTTP GET request
r = requests.get(endpoint, parameters, auth=(client_id, ''))
# Extract JSON data
json = r.json()

In [ ]:
# Check if the request worked, print out any errors
if r.status_code == 200:
    data = json['data']
    print('Data retrieved from frost.met.no!')
else:
    print('Error! Returned status code %s' % r.status_code)
    print('Message: %s' % json['error']['message'])
    print('Reason: %s' % json['error']['reason'])

The data comes back as a list with one entry pr day. Each entry contains a list of observations. The next cells will turn this into a pd.Dataframe. The MET version is using an older code, so pd.concat() will be used insteand of df.append()

In [ ]:
rows = []

for i in range(len(data)):
    row = pd.DataFrame(data[i]['observations'])
    row['referenceTime'] = data[i]['referenceTime']
    rows.append(row)

df = pd.concat(rows, ignore_index=True)

# Convert the time column to datetime and keep only the columns we need
df['referenceTime'] = pd.to_datetime(df['referenceTime'], utc=True).dt.tz_convert(None)
df = df[['referenceTime', 'value', 'timeOffset']]

# Check whether there are several measurements per day
print(df['timeOffset'].value_counts())

In [ ]:
# Keep one measurement per day and check the number of days
df = df[df['timeOffset'] == df['timeOffset'].iloc[0]].reset_index(drop=True)
print('Number of days: %s' % len(df))

In [ ]:
# Plot the daily mean temperature
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df['referenceTime'], df['value'], color='tab:red', linewidth=1.2)
ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')

ax.set_title('Daily mean air temperature in Ås (SN17850), 2025')
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
# Summary table with mean, median, min and max temperature
summary = pd.DataFrame({
    'Temperature 2025 (°C)': [
        df['value'].mean(),
        df['value'].median(),
        df['value'].min(),
        df['value'].max(),
    ]
}, index=['Mean', 'Median', 'Min', 'Max']).round(1)

print(summary)

## References

[1] Norwegian Meteorological Institute. *Frost – Python example*. https://frost.met.no/python_example.html (accessed 13 October 2026)

[2] Norwegian Meteorological Institute. *Frost API*. https://frost.met.no (accessed 13 October 2026). Weather data from station SN17850, Ås.

In [11]:
print(Path.cwd())
print(Path('frost_client_id.txt').exists())
print(len(client_id))

/Users/oskarrandel/Library/CloudStorage/OneDrive-NorwegianUniversityofLifeSciences/Deliverables/deliverable_1
True
0
